In [5]:
"""
check_gately_regularity_causal_shap.py
=======================================
Experimental regularity check for the Gately point, using the actual
coalition value function v(S) from the Causal SHAP pipeline.

  v(S | x_eval) = mean_{z ~ X_bg} f( x_eval_S , z_{-S} )

Checks per trial:
  Regularity  : x_i >= v({i})  for all i
  Efficiency  : sum(x_i) == v(N)

Runs on all 6 thesis datasets (real data). Falls back to synthetic
datasets if real data files are not found.

Usage:
  python check_gately_regularity_causal_shap.py [--data_dir /path/to/datasets]

Expects:
  gt_concepts/gately_with_checks.py   (Gately implementation)
"""

import sys, os, argparse
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Args ──────────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", default="../datasets")
parser.add_argument("--gt_dir",   default="gt_concepts",
                    help="Path to gt_concepts/ folder containing gately_with_checks.py")
parser.add_argument("--n_eval",   type=int, default=20,
                    help="x_eval points per dataset")
parser.add_argument("--n_bg",     type=int, default=80,
                    help="Background samples for v(S)")
parser.add_argument("--seed",     type=int, default=42)
args, _ = parser.parse_known_args()

# ── Stubs matching gately_with_checks.py interface ───────────────────────────
class _Data:
    def __init__(self, names): self.data_names = names
    def __len__(self):         return len(self.data_names)

class ValuationResult:
    def __init__(self, **kw): self.__dict__.update(kw)

class Utility:
    """Wraps v: frozenset -> float for the Gately interface."""
    def __init__(self, v_func, feat_names):
        self.data = _Data(feat_names)
        self._v   = v_func
    def __call__(self, S): return self._v(frozenset(S))

# ── Import Gately ─────────────────────────────────────────────────────────────
# gately_with_checks.py does `from .least_core import ValuationResult`
# We inject a fake least_core module into sys.modules so the relative
# import resolves to our stub above without needing the full package.
import types, sys as _sys
_lc = types.ModuleType("gt_concepts.least_core")
_lc.ValuationResult = ValuationResult
_sys.modules["gt_concepts.least_core"] = _lc
_sys.modules["gt_concepts"] = types.ModuleType("gt_concepts")

_sys.path.insert(0, args.gt_dir)
try:
    import importlib.util as _ilu
    _spec = _ilu.spec_from_file_location(
        "gt_concepts.gately_with_checks",
        os.path.join(args.gt_dir, "gately_with_checks.py"),
        submodule_search_locations=[]
    )
    _mod = _ilu.module_from_spec(_spec)
    _mod.__package__ = "gt_concepts"
    _spec.loader.exec_module(_mod)
    exact_gately_value = _mod.exact_gately_value
    print("[OK] Loaded exact_gately_value from gt_concepts/gately_with_checks.py\n")
except Exception as e:
    print(f"[ERROR] Could not load gt_concepts/gately_with_checks.py: {e}")
    import traceback; traceback.print_exc()
    _sys.exit(1)


#  DATASET LOADERS  (copied verbatim from causal_shap_drbo.py)

def _npz(fname):
    path = os.path.join(args.data_dir, fname)
    d    = np.load(path, allow_pickle=True)
    X    = np.concatenate([d['X_train'], d['X_test']], axis=0)
    y    = np.concatenate([d['y_train'], d['y_test']], axis=0)
    return X, y, [str(f) for f in d['feature_names']]

def load_soil():
    X, y, fn = _npz("soil_spi_preprocessed.npz")
    return X, y, fn, "multiclass", "Soil SPI"

def load_yield():
    X, y, fn = _npz("yield_preprocessed.npz")
    return X, y, fn, "multiclass", "Crop Yield"

def load_cv_risk():
    feat = ['diet_score','sleep_duration','family_history','bmi']
    df   = pd.read_csv(os.path.join(args.data_dir, "cv_risk.csv"))
    df   = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
    return df[feat].values.astype(float), df['cv_risk'].values.astype(float), \
           feat, "regression", "CV Risk"

def load_lung_cancer():
    feat = ['smoking','stress','drink_coffee']
    df   = pd.read_csv(os.path.join(args.data_dir, "lung_cancer_risk.csv"))
    df   = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
    return df[feat].values.astype(float), df['lung_cancer_risk'].values.astype(float), \
           feat, "regression", "Lung Cancer"

def load_wine():
    X, y, fn = _npz("wine_preprocessed.npz")
    return X, y, fn, "multiclass", "Wine"

def load_compas():
    X, y, fn = _npz("compas_preprocessed.npz")
    return X, y, fn, "binary", "COMPAS"

REAL_LOADERS = [load_soil, load_yield, load_cv_risk,
                load_lung_cancer, load_wine, load_compas]


#  SYNTHETIC FALLBACK DATASETS (used if real files not found)

def _synth_regression(n=300, d=4, seed=10):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    w   = rng.uniform(0.5, 2.5, d) * rng.choice([-1,1], d)
    y   = X @ w + rng.standard_normal(n) * 0.3
    return X, y, [f"x{i}" for i in range(d)], "regression", f"Synth-Reg-d{d}"

def _synth_binary(n=300, d=4, seed=20):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    w   = rng.uniform(0.5, 2.0, d)
    lp  = X @ w
    y   = (lp > lp.mean()).astype(int)
    return X, y, [f"x{i}" for i in range(d)], "binary", f"Synth-Bin-d{d}"

def _synth_multiclass(n=300, d=5, seed=30):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    y   = (X[:,0]+X[:,1]>0).astype(int) + (X[:,2]>0.5).astype(int)
    return X, y, [f"x{i}" for i in range(d)], "multiclass", f"Synth-MC-d{d}"

SYNTH_DATASETS = [
    _synth_regression(d=4, seed=10), _synth_regression(d=6, seed=11),
    _synth_binary(d=4, seed=20),     _synth_binary(d=5, seed=21),
    _synth_multiclass(d=5, seed=30), _synth_multiclass(d=7, seed=31),
]


#  COALITION VALUE FUNCTION  - mirrors causal_shap_drbo.py exactly

def make_v(x_eval, X_bg, predict_fn):
    def v(S):
        S = list(S)
        if not S:
            return float(predict_fn(X_bg).mean())
        Z = X_bg.copy()
        for f in S:
            Z[:, f] = x_eval[f]
        return float(predict_fn(Z).mean())
    return v

def build_predict(X, y, task, seed=42):
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    if task == "regression":
        m = LinearRegression().fit(Xs, y)
        return lambda Z: m.predict(sc.transform(Z))
    elif task == "binary":
        m = LogisticRegression(max_iter=1000, C=1.0, random_state=seed).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z))[:,1]
    else:
        m = LogisticRegression(max_iter=1000, C=1.0, random_state=seed).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z)).max(axis=1)


# Single trial

def run_trial(X, y, task, feat_names, x_eval, n_bg, seed):
    d       = len(feat_names)
    rng     = np.random.default_rng(seed)
    bg_idx  = rng.choice(len(X), size=min(n_bg, len(X)), replace=False)
    predict = build_predict(X, y, task, seed=seed)
    v       = make_v(x_eval, X[bg_idx], predict)
    utility = Utility(v, feat_names)

    try:
        res     = exact_gately_value(utility)
        x_alloc = res.values
        v_i     = np.array([v(frozenset([i])) for i in range(d)])
        vN      = v(frozenset(range(d)))
        reg_ok  = np.all(x_alloc >= v_i - 1e-9)
        eff_ok  = np.isclose(x_alloc.sum(), vN, atol=1e-6)
        ok      = reg_ok and eff_ok
        return ('pass' if ok else 'fail'), x_alloc, v_i, vN, None
    except ValueError as e:
        return 'skip', None, None, None, str(e)

    # (unreachable - kept for structure)


# Per-dataset runner

def run_dataset(X, y, feat_names, task, name):
    rng = np.random.default_rng(args.seed + hash(name) % 1000)
    d   = len(feat_names)
    p = f = s = 0
    fail_details = []

    for trial in range(args.n_eval):
        x_eval = X[rng.integers(len(X))]
        trial_seed = int(rng.integers(100000))
        status, x_alloc, v_i, vN, err = run_trial(
            X, y, task, feat_names, x_eval, args.n_bg, trial_seed)

        if   status == 'pass': p += 1
        elif status == 'fail':
            f += 1
            fail_details.append((trial, x_alloc, v_i, vN))
        else:
            s += 1
            if s <= 3:   # print first 3 skips per dataset to avoid spam
                d_  = len(feat_names)
                rng2 = np.random.default_rng(trial_seed)
                bg2  = rng2.choice(len(X), size=min(args.n_bg, len(X)), replace=False)
                pf   = build_predict(X, y, task, seed=trial_seed)
                v_   = make_v(x_eval, X[bg2], pf)
                v_i_ = np.array([v_(frozenset([i])) for i in range(d_)])
                vN_  = v_(frozenset(range(d_)))
                M_i_ = np.array([vN_ - v_(frozenset([j for j in range(d_) if j != i]))
                                  for i in range(d_)])
                print(f"\n    [SKIP trial={trial}] reason: {err}")
                print(f"      v(N)       = {vN_:.6f}")
                print(f"      sum(v_i)   = {v_i_.sum():.6f}  {'<= v(N) OK' if v_i_.sum() <= vN_+1e-9 else '> v(N) ESSENTIAL VIOLATED'}")
                print(f"      sum(M_i)   = {M_i_.sum():.6f}  {'>= v(N) OK' if M_i_.sum() >= vN_-1e-9 else '< v(N) SUPERADDITIVE VIOLATED'}")
                print(f"      v_i        = {np.round(v_i_, 4)}")
                print(f"      M_i        = {np.round(M_i_, 4)}")

    print(f"\n  Dataset        : {name}")
    print(f"  task / d       : {task} / {d}")
    print(f"  n_samples      : {len(X)}")
    print(f"  eval points    : {args.n_eval}  (n_bg={args.n_bg})")
    print(f"  Valid games    : {p+f}  |  PASS: {p}  FAIL: {f}  SKIP(not essential/std): {s}")

    for trial, x_alloc, v_i, vN in fail_details:
        print(f"\n    !! FAIL trial={trial}")
        print(f"       features  : {feat_names}")
        print(f"       x_alloc   : {np.round(x_alloc, 5)}")
        print(f"       v_i       : {np.round(v_i, 5)}")
        print(f"       x_i-v_i   : {np.round(x_alloc-v_i, 7)}")
        print(f"       sum(x)={x_alloc.sum():.6f}  v(N)={vN:.6f}")

    return p, f, s


# Main

def main():
    print("="*70)
    print("  Gately Regularity Check  |  Causal SHAP Coalition Game")
    print(f"  v(S) = E_bg[f(x_eval_S, z_{{-S}})]   (background marginalisation)")
    print(f"  n_eval={args.n_eval}  n_bg={args.n_bg}  seed={args.seed}")
    print("="*70)

    # ── Try loading real datasets; fall back to synthetic if missing ──────────
    datasets = []
    any_real = False
    for loader in REAL_LOADERS:
        try:
            entry = loader()
            datasets.append(entry)
            any_real = True
            print(f"  [LOADED] {entry[4]}  n={len(entry[0])}  d={len(entry[2])}")
        except Exception as e:
            print(f"  [SKIP]   {loader.__name__}: {e}")

    if not any_real:
        print("\n  No real datasets found — running on synthetic fallbacks.\n")
        datasets = SYNTH_DATASETS
    elif len(datasets) < len(REAL_LOADERS):
        print(f"\n  {len(datasets)}/{len(REAL_LOADERS)} real datasets loaded."
              f" Skipped datasets will not be tested.\n")

    print()
    grand_p = grand_f = grand_s = 0
    for X, y, feat_names, task, name in datasets:
        p, f, s = run_dataset(X, y, feat_names, task, name)
        grand_p += p; grand_f += f; grand_s += s

    print("\n" + "="*70)
    print("  GRAND TOTALS  across all datasets")
    print(f"  PASS : {grand_p}   FAIL : {grand_f}   SKIP : {grand_s}")
    print("  Verdict :", "REGULARITY HOLDS in all valid games ok"
                         if grand_f == 0
                         else f"!! {grand_f} VIOLATION(S) FOUND")
    print("="*70 + "\n")

if __name__ == "__main__":
    main()

[OK] Loaded exact_gately_value from gt_concepts/gately_with_checks.py

  Gately Regularity Check  |  Causal SHAP Coalition Game
  v(S) = E_bg[f(x_eval_S, z_{-S})]   (background marginalisation)
  n_eval=20  n_bg=80  seed=42
  [LOADED] Soil SPI  n=5000  d=7
  [LOADED] Crop Yield  n=1000  d=4
  [LOADED] CV Risk  n=1000  d=4
  [LOADED] Lung Cancer  n=1000  d=3
  [LOADED] Wine  n=177  d=13
  [LOADED] COMPAS  n=5278  d=19

[0.81918655 0.81875454 0.8196738  0.82328678 0.82493532 0.82781351
 0.85260487]

    [SKIP trial=0] reason: Game is not Essential: sum(v_i)=5.7863 <= v(N)=0.9167 <= sum(M_i)=0.4989 is violated.
      v(N)       = 0.916676
      sum(v_i)   = 5.786255  > v(N) ESSENTIAL VIOLATED
      sum(M_i)   = 0.498944  < v(N) SUPERADDITIVE VIOLATED
      v_i        = [0.8192 0.8188 0.8197 0.8233 0.8249 0.8278 0.8526]
      M_i        = [0.0924 0.0606 0.0222 0.1564 0.1054 0.0026 0.0595]
[0.8311565  0.87443085 0.85414055 0.82736398 0.80357313 0.86228231
 0.82758364]

    [SKIP trial=1] re

In [6]:
"""
check_gately_regularity_causal_shap.py
=======================================
Experimental regularity check for the Gately point, using the actual
coalition value function v(S) from the Causal SHAP pipeline.

  v(S | x_eval) = mean_{z ~ X_bg} f( x_eval_S , z_{-S} )

Checks per trial:
  Regularity  : x_i >= v({i})  for all i
  Efficiency  : sum(x_i) == v(N)

Runs on all 6 thesis datasets (real data). Falls back to synthetic
datasets if real data files are not found.

Usage:
  python check_gately_regularity_causal_shap.py [--data_dir /path/to/datasets]

Expects:
  gt_concepts/gately_with_checks.py   (Gately implementation)
"""

import sys, os, argparse
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Args ──────────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", default="../datasets")
parser.add_argument("--gt_dir",   default="gt_concepts",
                    help="Path to gt_concepts/ folder containing gately_with_checks.py")
parser.add_argument("--n_eval",   type=int, default=20,
                    help="x_eval points per dataset")
parser.add_argument("--n_bg",     type=int, default=80,
                    help="Background samples for v(S)")
parser.add_argument("--seed",     type=int, default=42)
args, _ = parser.parse_known_args()

# ── Stubs matching gately_with_checks.py interface ───────────────────────────
class _Data:
    def __init__(self, names): self.data_names = names
    def __len__(self):         return len(self.data_names)

class ValuationResult:
    def __init__(self, **kw): self.__dict__.update(kw)

class Utility:
    """Wraps v: frozenset -> float for the Gately interface."""
    def __init__(self, v_func, feat_names):
        self.data = _Data(feat_names)
        self._v   = v_func
    def __call__(self, S): return self._v(frozenset(S))

# ── Import Gately ─────────────────────────────────────────────────────────────
# gately_with_checks.py does `from .least_core import ValuationResult`
# We inject a fake least_core module into sys.modules so the relative
# import resolves to our stub above without needing the full package.
import types, sys as _sys
_lc = types.ModuleType("gt_concepts.least_core")
_lc.ValuationResult = ValuationResult
_sys.modules["gt_concepts.least_core"] = _lc
_sys.modules["gt_concepts"] = types.ModuleType("gt_concepts")

_sys.path.insert(0, args.gt_dir)
try:
    import importlib.util as _ilu
    _spec = _ilu.spec_from_file_location(
        "gt_concepts.gately_with_checks",
        os.path.join(args.gt_dir, "gately_with_checks.py"),
        submodule_search_locations=[]
    )
    _mod = _ilu.module_from_spec(_spec)
    _mod.__package__ = "gt_concepts"
    _spec.loader.exec_module(_mod)
    exact_gately_value = _mod.exact_gately_value
    print("[OK] Loaded exact_gately_value from gt_concepts/gately_with_checks.py\n")
except Exception as e:
    print(f"[ERROR] Could not load gt_concepts/gately_with_checks.py: {e}")
    import traceback; traceback.print_exc()
    _sys.exit(1)


#  DATASET LOADERS  (copied verbatim from causal_shap_drbo.py)

def _npz(fname):
    path = os.path.join(args.data_dir, fname)
    d    = np.load(path, allow_pickle=True)
    X    = np.concatenate([d['X_train'], d['X_test']], axis=0)
    y    = np.concatenate([d['y_train'], d['y_test']], axis=0)
    return X, y, [str(f) for f in d['feature_names']]

def load_soil():
    X, y, fn = _npz("soil_spi_preprocessed.npz")
    return X, y, fn, "multiclass", "Soil SPI"

def load_yield():
    X, y, fn = _npz("yield_preprocessed.npz")
    return X, y, fn, "multiclass", "Crop Yield"

def load_cv_risk():
    feat = ['diet_score','sleep_duration','family_history','bmi']
    df   = pd.read_csv(os.path.join(args.data_dir, "cv_risk.csv"))
    df   = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
    return df[feat].values.astype(float), df['cv_risk'].values.astype(float), \
           feat, "regression", "CV Risk"

def load_lung_cancer():
    feat = ['smoking','stress','drink_coffee']
    df   = pd.read_csv(os.path.join(args.data_dir, "lung_cancer_risk.csv"))
    df   = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
    return df[feat].values.astype(float), df['lung_cancer_risk'].values.astype(float), \
           feat, "regression", "Lung Cancer"

def load_wine():
    X, y, fn = _npz("wine_preprocessed.npz")
    return X, y, fn, "multiclass", "Wine"

def load_compas():
    X, y, fn = _npz("compas_preprocessed.npz")
    return X, y, fn, "binary", "COMPAS"

REAL_LOADERS = [load_soil, load_yield, load_cv_risk,
                load_lung_cancer, load_wine, load_compas]


#  SYNTHETIC FALLBACK DATASETS (used if real files not found)

def _synth_regression(n=300, d=4, seed=10):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    w   = rng.uniform(0.5, 2.5, d) * rng.choice([-1,1], d)
    y   = X @ w + rng.standard_normal(n) * 0.3
    return X, y, [f"x{i}" for i in range(d)], "regression", f"Synth-Reg-d{d}"

def _synth_binary(n=300, d=4, seed=20):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    w   = rng.uniform(0.5, 2.0, d)
    lp  = X @ w
    y   = (lp > lp.mean()).astype(int)
    return X, y, [f"x{i}" for i in range(d)], "binary", f"Synth-Bin-d{d}"

def _synth_multiclass(n=300, d=5, seed=30):
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, d))
    y   = (X[:,0]+X[:,1]>0).astype(int) + (X[:,2]>0.5).astype(int)
    return X, y, [f"x{i}" for i in range(d)], "multiclass", f"Synth-MC-d{d}"

SYNTH_DATASETS = [
    _synth_regression(d=4, seed=10), _synth_regression(d=6, seed=11),
    _synth_binary(d=4, seed=20),     _synth_binary(d=5, seed=21),
    _synth_multiclass(d=5, seed=30), _synth_multiclass(d=7, seed=31),
]


#  COALITION VALUE FUNCTION  - mirrors causal_shap_drbo.py exactly

def make_v(x_eval, X_bg, predict_fn):
    v_empty = float(predict_fn(X_bg).mean())
    def v(S):
        S = list(S)
        if not S:
            return 0.0
        Z = X_bg.copy()
        for f in S:
            Z[:, f] = x_eval[f]
        return float(predict_fn(Z).mean()) - v_empty   # centered
    return v

def build_predict(X, y, task, seed=42):
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    if task == "regression":
        m = LinearRegression().fit(Xs, y)
        return lambda Z: m.predict(sc.transform(Z))
    elif task == "binary":
        m = LogisticRegression(max_iter=1000, C=1.0, random_state=seed).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z))[:,1]
    else:
        m = LogisticRegression(max_iter=1000, C=1.0, random_state=seed).fit(Xs, y)
        return lambda Z: m.predict_proba(sc.transform(Z)).max(axis=1)


# Single trial

def run_trial(X, y, task, feat_names, x_eval, n_bg, seed):
    d       = len(feat_names)
    rng     = np.random.default_rng(seed)
    bg_idx  = rng.choice(len(X), size=min(n_bg, len(X)), replace=False)
    predict = build_predict(X, y, task, seed=seed)
    v       = make_v(x_eval, X[bg_idx], predict)
    utility = Utility(v, feat_names)

    try:
        res     = exact_gately_value(utility)
        x_alloc = res.values
        v_i     = np.array([v(frozenset([i])) for i in range(d)])
        vN      = v(frozenset(range(d)))
        reg_ok  = np.all(x_alloc >= v_i - 1e-9)
        eff_ok  = np.isclose(x_alloc.sum(), vN, atol=1e-6)
        ok      = reg_ok and eff_ok
        return ('pass' if ok else 'fail'), x_alloc, v_i, vN, None
    except ValueError as e:
        return 'skip', None, None, None, str(e)

    # (unreachable - kept for structure)


# Per-dataset runner

def run_dataset(X, y, feat_names, task, name):
    rng = np.random.default_rng(args.seed + hash(name) % 1000)
    d   = len(feat_names)
    p = f = s = 0
    fail_details = []

    for trial in range(args.n_eval):
        x_eval = X[rng.integers(len(X))]
        trial_seed = int(rng.integers(100000))
        status, x_alloc, v_i, vN, err = run_trial(
            X, y, task, feat_names, x_eval, args.n_bg, trial_seed)

        if   status == 'pass': p += 1
        elif status == 'fail':
            f += 1
            fail_details.append((trial, x_alloc, v_i, vN))
        else:
            s += 1
            if s <= 3:   # print first 3 skips per dataset to avoid spam
                d_  = len(feat_names)
                rng2 = np.random.default_rng(trial_seed)
                bg2  = rng2.choice(len(X), size=min(args.n_bg, len(X)), replace=False)
                pf   = build_predict(X, y, task, seed=trial_seed)
                v_   = make_v(x_eval, X[bg2], pf)
                v_i_ = np.array([v_(frozenset([i])) for i in range(d_)])
                vN_  = v_(frozenset(range(d_)))
                M_i_ = np.array([vN_ - v_(frozenset([j for j in range(d_) if j != i]))
                                  for i in range(d_)])
                print(f"\n    [SKIP trial={trial}] reason: {err}")
                print(f"      v(N)       = {vN_:.6f}")
                print(f"      sum(v_i)   = {v_i_.sum():.6f}  {'<= v(N) OK' if v_i_.sum() <= vN_+1e-9 else '> v(N) ESSENTIAL VIOLATED'}")
                print(f"      sum(M_i)   = {M_i_.sum():.6f}  {'>= v(N) OK' if M_i_.sum() >= vN_-1e-9 else '< v(N) SUPERADDITIVE VIOLATED'}")
                print(f"      v_i        = {np.round(v_i_, 4)}")
                print(f"      M_i        = {np.round(M_i_, 4)}")

    print(f"\n  Dataset        : {name}")
    print(f"  task / d       : {task} / {d}")
    print(f"  n_samples      : {len(X)}")
    print(f"  eval points    : {args.n_eval}  (n_bg={args.n_bg})")
    print(f"  Valid games    : {p+f}  |  PASS: {p}  FAIL: {f}  SKIP(not essential/std): {s}")

    for trial, x_alloc, v_i, vN in fail_details:
        print(f"\n    !! FAIL trial={trial}")
        print(f"       features  : {feat_names}")
        print(f"       x_alloc   : {np.round(x_alloc, 5)}")
        print(f"       v_i       : {np.round(v_i, 5)}")
        print(f"       x_i-v_i   : {np.round(x_alloc-v_i, 7)}")
        print(f"       sum(x)={x_alloc.sum():.6f}  v(N)={vN:.6f}")

    return p, f, s


# Main

def main():
    print("="*70)
    print("  Gately Regularity Check  |  Causal SHAP Coalition Game")
    print(f"  v(S) = E_bg[f(x_eval_S, z_{{-S}})]   (background marginalisation)")
    print(f"  n_eval={args.n_eval}  n_bg={args.n_bg}  seed={args.seed}")
    print("="*70)

    # ── Try loading real datasets; fall back to synthetic if missing ──────────
    datasets = []
    any_real = False
    for loader in REAL_LOADERS:
        try:
            entry = loader()
            datasets.append(entry)
            any_real = True
            print(f"  [LOADED] {entry[4]}  n={len(entry[0])}  d={len(entry[2])}")
        except Exception as e:
            print(f"  [SKIP]   {loader.__name__}: {e}")

    if not any_real:
        print("\n  No real datasets found — running on synthetic fallbacks.\n")
        datasets = SYNTH_DATASETS
    elif len(datasets) < len(REAL_LOADERS):
        print(f"\n  {len(datasets)}/{len(REAL_LOADERS)} real datasets loaded."
              f" Skipped datasets will not be tested.\n")

    print()
    grand_p = grand_f = grand_s = 0
    for X, y, feat_names, task, name in datasets:
        p, f, s = run_dataset(X, y, feat_names, task, name)
        grand_p += p; grand_f += f; grand_s += s

    print("\n" + "="*70)
    print("  GRAND TOTALS  across all datasets")
    print(f"  PASS : {grand_p}   FAIL : {grand_f}   SKIP : {grand_s}")
    print("  Verdict :", "REGULARITY HOLDS in all valid games ok"
                         if grand_f == 0
                         else f"!! {grand_f} VIOLATION(S) FOUND")
    print("="*70 + "\n")

if __name__ == "__main__":
    main()

[OK] Loaded exact_gately_value from gt_concepts/gately_with_checks.py

  Gately Regularity Check  |  Causal SHAP Coalition Game
  v(S) = E_bg[f(x_eval_S, z_{-S})]   (background marginalisation)
  n_eval=20  n_bg=80  seed=42
  [LOADED] Soil SPI  n=5000  d=7
  [LOADED] Crop Yield  n=1000  d=4
  [LOADED] CV Risk  n=1000  d=4
  [LOADED] Lung Cancer  n=1000  d=3
  [LOADED] Wine  n=177  d=13
  [LOADED] COMPAS  n=5278  d=19


    [SKIP trial=0] reason: Game is not Semi-Standard: v({i}) > M_i(v) for players [5].
      v(N)       = 0.097819
      sum(v_i)   = 0.054251  <= v(N) OK
      sum(M_i)   = 0.498944  >= v(N) OK
      v_i        = [ 0.0003 -0.0001  0.0008  0.0044  0.0061  0.009   0.0337]
      M_i        = [0.0924 0.0606 0.0222 0.1564 0.1054 0.0026 0.0595]
[-0.02215182  0.02112253  0.00083223 -0.02594435 -0.04973519  0.00897399
 -0.02572469]

    [SKIP trial=1] reason: Game is not Essential: sum(v_i)=-0.0926 <= v(N)=-0.1644 <= sum(M_i)=-0.5310 is violated.
      v(N)       = -0.164389
  